# PG-RAD to PyMC pipeline -- Source separation

This notebook is an automation pipeline to sample two-source geometries of varying source separations.

The script first generates scenarios using PG-RAD with source separation in FWHM as a variable (precomputed for the given distance to the road)

The generated data is passed into the Bayesian pipeline, and 10 simulations are done. The traces are written to disk and can be analyzed in the `trace-analysis.ipynb` notebook.

### Parameters

The `DETECTOR` and `DET_EFF` parameters will be used for the Bayesian part, and need to be set correctly.

In [1]:
TRACE_OUT_FOLDER = "trace_output/"
SIM_NAME = 'fwhm_NaI'

#DETECTOR = "HPGe"
#DET_EFF = 0.002410082036721662

DETECTOR = "NaIR"
DET_NAME_PGRAD = "LU_NaIR"
DET_EFF = 0.0261

DIST_TO_ROAD = 70 # meters
ACT_SRC = 300 # MBq
MU_AIR = 0.009636447511845

d = [4.0]# , 2.0, 1.5, 1.2, 1] # [4.0, 3.5, 3.0, 2.5, 2.0, 1.95, 1.9, 1.85, 1.8, 1.75, 1.7, 1.65, 1.6, 1.55, 1.5, 1.45, 1.4, 1.35, 1.3, 1.25, 1.2, 1.15, 1.1, 1.05, 1.0] # multiples of the FWHM

## Imports

In [2]:
from pathlib import Path
import json
import time
import sys

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit, brentq
import yaml
import arviz as az

from matplotlib import pyplot as plt
import matplotlib.patches as patches

from pg_rad.utils.projection import rel_to_abs_source_position as convert_pos

# src directory
parent_dir = Path.cwd().parent.parent 
sys.path.append(str(parent_dir))
print(parent_dir)

from run_model import run

/home/pim/pg-rad-analysis/src


## Helper function

To load the simulations for Bayesian inference

In [3]:
def get_sim_path_and_csv_name(i, base_dir="output"):
    base_path = Path(base_dir)
    # Find all directories matching the pattern f"{i}_result_*"
    folders = list(base_path.glob(f"{i}_result_*"))
    if not folders:
        raise FileNotFoundError(f"No folder matching {i}_result_* found in {base_dir}")

    folder_name = folders[0].name
    print("Detected folder name:", folder_name)

    # Find all CSV files in the folder
    csv_files = list(folders[0].glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV file found in {folders[0]}")

    csv_name = csv_files[0].name
    print("Detected CSV name:", csv_name)

    return str(folders[0]), csv_name

def expected_fwhm(distance_to_road_m, activity_mbq, mu_air=0.0):
    r = float(distance_to_road_m)
    peak = np.exp(-mu_air * r) / (r**2)
    half_peak = peak / 2

    def signal(x):
        d = np.sqrt(x**2 + r**2)
        return np.exp(-mu_air * d) / (d**2)

    def equation(x):
        return signal(x) - half_peak

    x_half = brentq(equation, 0, 1000 * r)
    return 2 * x_half

## Loading the base configuration

Below the base config for source separation scenario is loaded. The parameters regarding source geometry will be modified dynamically using the JSON library of Python.

In [4]:
with open('base_config.yml') as f:
    conf = yaml.safe_load(f)
conf['detector'] = DET_NAME_PGRAD
conf

{'name': 'test',
 'speed': 8.33,
 'acquisition_time': 1,
 'path': {'length': [100, 2000], 'segments': [{'turn_left': 30}, 'straight']},
 'sources': {'s1': {'activity_MBq': 300,
   'isotope': 'Cs137',
   'gamma_energy_keV': 662,
   'position': [0, 0, 0]},
  's2': {'activity_MBq': 300,
   'isotope': 'Cs137',
   'gamma_energy_keV': 662,
   'position': [0, 0, 0]}},
 'detector': 'LU_NaIR',
 'options': {'bkg_cps': 0}}

## Simulation parameters

In [5]:
FWHM = expected_fwhm(DIST_TO_ROAD, ACT_SRC, mu_air=MU_AIR)
FWHM

112.91622089397218

We create a DataFrame containing all the scenarios

In [6]:
df = pd.DataFrame(columns=["d", "A1", "A2", "r1", "r2", "A1/A2", "r1/r2"])
df['d'] = np.array(d)
df['A1'] = np.full_like(df['d'], ACT_SRC)
df['A2']= np.full_like(df['d'], ACT_SRC)
df['A1/A2'] = df['A1'] / df['A2'] 
df['r1'] = np.full_like(df['d'], DIST_TO_ROAD)
df['r2'] = np.full_like(df['d'], DIST_TO_ROAD)
df['r1/r2'] = df['r1'] / df['r2'] 
df.head()

,d,A1,A2,r1,r2,A1/A2,r1/r2
0,4.0,300.0,300.0,70.0,70.0,1.0,1.0


## Generating the scenarios using PG-RAD

Below is a for loop which will iterate over the rows of the above DataFrame and its configurations, dynamically write a new PG-RAD configuration, generate the scenario and save the output.

In [7]:
conf_list = []

midpoint_of_road = (sum(conf['path']['length']) - 100) / 2
for index, row in df.iterrows():
    d = row['d'] * FWHM
    conf['sources']['s1']['activity_MBq'] = float(row['A1'])
    conf['sources']['s2']['activity_MBq'] = float(row['A2'])
    conf['sources']['s1']['position'] = {'along_path': float(midpoint_of_road - d/2), 'dist_from_path': float(row['r1']), 'side': 'left'}
    conf['sources']['s2']['position'] = {'along_path': float(midpoint_of_road + d/2), 'dist_from_path': float(row['r2']), 'side': 'left'}
    conf['name'] = str(row['d'])+f'_{SIM_NAME}'

    with open('tmp.yml', 'w') as f:
        yaml.dump(conf, f, default_flow_style=False)
    
    ! cd pgrad_output && pgrad --config ../tmp.yml --save

2026-05-16 12:50:30,226 - INFO: Landscape built successfully: 4.0_fwhm_NaI
<class 'numpy.bool'>
2026-05-16 12:50:30,247 - INFO: Simulation output saved to 4.0_fwhm_nai_result_20260516_1250!


## Bayesian inference

In [8]:
for i in df['d']:
    csv_path, roi_filename = get_sim_path_and_csv_name(str(i)+'_'+SIM_NAME.lower(), base_dir='pgrad_output')

    BASE_DIR = Path("__file__").resolve().parent
    CSV_DIR = Path(BASE_DIR.joinpath(csv_path))

    params_file = Path(CSV_DIR.joinpath("parameters.json"))
    params = json.load(params_file.open())

    csv_file = Path(CSV_DIR.joinpath(roi_filename))
    df_i = pd.read_csv(csv_file)

    df_i['ROI_BR'] += 90
    df_i['ROI_P'] += df_i['ROI_BR']

    

    for j in range(10): # 10 to compute localization prob.
        trace, real_params = run(df_i, params, csv_file, DETECTOR, DET_EFF)
        az.to_netcdf(trace, filename=TRACE_OUT_FOLDER+str(i)+'_'+SIM_NAME+"_trace_"+str(j))
        df_i.to_pickle(TRACE_OUT_FOLDER+str(i)+'_'+SIM_NAME+"_pkl_"+str(j))
    
        with open(TRACE_OUT_FOLDER+str(i)+'_'+SIM_NAME+"_real_params_"+str(j),'w') as fp:
            json.dump(real_params, fp)

Detected folder name: 4.0_fwhm_nai_result_20260515_1333
Detected CSV name: 1_2_src_0_cps_bkg_300MBq_300MBq_70m_70m_652_425_1043_651.csv


/home/pim/miniforge3/envs/pg-rad-analysis/lib/python3.12/site-packages/pymc/sampling/mcmc.py:786: UserWarning: A list or tuple of random_seed no longer specifies the specific random_seed of each chain. Use a single seed instead.
  warnings.warn(
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [y_src, x_src, act_src]


Output()

Sampling 2 chains for 500 tune and 2_000 draw iterations (1_000 + 4_000 draws total) took 5 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Sampling: [predicted_cps]


Output()

...

Output()

/home/pim/miniforge3/envs/pg-rad-analysis/lib/python3.12/site-packages/pymc/sampling/mcmc.py:786: UserWarning: A list or tuple of random_seed no longer specifies the specific random_seed of each chain. Use a single seed instead.
  warnings.warn(
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [y_src, x_src, act_src]


Output()

Sampling 2 chains for 500 tune and 2_000 draw iterations (1_000 + 4_000 draws total) took 5 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Sampling: [predicted_cps]


Output()